# **STAGE 3 - MACHINE LEARNING**

## **Objectives**

* Write your notebook objective here, for example, "Fetch data from Kaggle and save as raw data", or "engineer features for modelling"

## **Inputs**

* Write down which data or information you need to run the notebook 

## **Outputs**

* Write here which files, code or artefacts you generate by the end of the notebook 

## **Additional Comments**

* If you have any additional comments that don't fit in the previous bullets, please state them here. 



---

## **Change working directory**

* We are assuming you will store the notebooks in a subfolder, therefore when running the notebook in the editor, you will need to change the working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [1]:
import os
current_dir = os.getcwd()
current_dir

'/Users/elliebrawn/Documents/vscode-projects/student-performance-analysis/jupyter_notebooks'

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chir() defines the new current directory

In [2]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

You set a new current directory


Confirm the new current directory

In [3]:
current_dir = os.getcwd()
current_dir

'/Users/elliebrawn/Documents/vscode-projects/student-performance-analysis'

---

## **Import Packages and Libraries**

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")

# Import Scikit-learn Pipeline
from sklearn.pipeline import Pipeline

# Import ColumnTransformer, which is used to apply different preprocessing steps to different columns of the dataset
from sklearn.compose import ColumnTransformer

# Import OneHotEncoder, which is used to convert categorical variables into binary vectors (one-hot encoding)
from sklearn.preprocessing import OneHotEncoder

# Import StandardScaler, which is used to standardise numerical data to make sure the data points have a balanced scale
from sklearn.preprocessing import StandardScaler

# Import SelectFromModel, which is used for feature selection based on the importance of features determined by a model
from sklearn.feature_selection import SelectFromModel

# Import ML Algorithm Models
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor

# Import regression metrics
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error 

---

## **Load Dataset**

First, I will load the cleaned dataset `student_performance_dataset_cleaned.csv`.

In [5]:
# Load the cleaned dataset
df_ml = pd.read_csv("datasets/cleaned-data/student_performance_dataset_cleaned.csv")

# Display the first few rows of the dataset to make sure the data has loaded correctly
df_ml.head()

,student_id,gender,study_time_hours,attendance_percent,sleep_hours,parental_education,internet_access,extracurricular_activities,part_time_job,previous_grade,final_exam_score,final_grade,final_grade_category,final_grade_binary
0,1,Male,4.0,98.0,6.5,Bachelors,Yes,Yes,No,76.9,100.0,A,High Grade,1
1,2,Female,6.3,100.0,5.7,High School,Yes,Yes,Yes,75.5,100.0,A,High Grade,1
2,3,Male,4.9,85.3,7.9,Bachelors,Yes,No,Yes,88.5,97.3,A,High Grade,1
3,4,Male,2.6,77.5,8.0,Unknown,Yes,Yes,No,85.1,83.8,B,High Grade,1
4,5,Male,2.2,89.6,4.6,Bachelors,Yes,No,Yes,61.8,68.3,D,Low Grade,0


---

## **1. Machine Learning Algorithm Selection**

### **1.1 Review the Target Variable**

The first thing I am going to do is decide what machine learning algorithms are going to be appropriate to predict the target.

| **What are you predicting?** | **Target Variable?** | **Algorithm Type** | **Examples** |
| ----- | ----- | ----- | ----- |
| Continuous Number | Yes | Regression Algorithm | Linear Regression, Decision Tree Regression, Random Forest Regression|
| Category | Yes | Classification Algorithm | Logistic Regression, Decision Tree Classification, Random Forest Classification |
| Category | No | Clustering Algorithm | K-Means Clustering |

In [9]:
# Verify what datatype the target variable is
df_ml["final_exam_score"].dtype

if df_ml["final_exam_score"].dtype == "object":
    print("The target variable is categorical.")
elif df_ml["final_exam_score"].dtype == "int64":
    print("The target variable is numerical.")
elif df_ml["final_exam_score"].dtype == "float64":
    print("The target variable is numerical.")
else:
    print("The target variable type is not expected.")

The target variable is numerical.


### **1.2 Algorithm Selection**

Given that the target variable is numerical, this gives a clear picture of what algorithms we can use.

In finding the best model to predict `final_exam_score`, I am going to test **Linear Regression**, **Decision Tree Regression** and **Random Forest Regression**.

---

## **2. Split the Dataset into Test and Train**

### **2.1 Define X and y**

Firstly, I am going to define X and y: in machine learning, X represents the input features and y represents the target.

For our model, `final_exam_score` is our chosen target variable - X. 

I have chosen to not include the following variables as features - y:
`student_id`: this acts as a label for each student and carries no insight beyond being a form of indexing.
`final_grade`: as explored in the EDA section of **Stage 2 - Visualisation** this is very closely correlated to our target variable and would lead to considerable *target leakage*
`final_grade_category` and `final_grade_binary`: these were groups created from `final_grade` in the feature engineering section of **Stage 1 - ETL** and would therefore also train the model with a direct indicator of the target. 

In [10]:
# Define X as the features that will be ussed to train the model
X = df_ml[[
    "gender",
    "study_time_hours",
    "attendance_percent",
    "sleep_hours",
    "parental_education",
    "internet_access",
    "extracurricular_activities",
    "part_time_job",
    "previous_grade"
]]

# Define y as the target variable that we want to predict
y = df_ml["final_exam_score"]

### **2.2 Split the Dataset**

To split the dataset, I am going to use `test_train_split` from scikit-learn (`sklearn`) to separate the data into a train set (80% of the overall dataset) and a test set (20% of the overall dataset).

In [11]:
# Use test_train_split to split the data into a test set (20%) and a train set (80%)
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"The dataset has been split into a Train set and a Test set.\n"
      f"Train set: {X_train.shape}, {y_train.shape}\n"
      f"Test set: {X_test.shape}, {y_test.shape}")

The dataset has been split into a Train set and a Test set.
Train set: (800, 9), (800,)
Test set: (200, 9), (200,)


> *Troubleshooting Issue*:
<br><br>When first splitting the test and train sets , I included `X=` and `y=` as parameters, which threw up an error. However, when I looked more closely at the documentation for train_test_split, I noted that these actually needed to just be arrays.
<br><br>Including all the features in a list (for `X` in particular) felt quite difficult to follow, so I decided instead to first **Define X and y** and then parse these two variables into the  `train_test_split()` method.

---